# Reliable RAG with Gemini

This notebook builds a simple manual RAG pipeline using Gemini embeddings and generation.

### Visual Representation

<img src="../images/reliable_rag.svg" alt="Reliable-RAG" width="300">

In [ ]:
# Setup Gemini and Environment
import os
import json
from typing import List, Dict

import numpy as np
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment or .env file")

genai.configure(api_key=GOOGLE_API_KEY)

llm_model_name = "gemini-2.0-flash"
embedding_model_name = "models/embedding-001"

### Create Vectorstore (Manual Implementation without LangChain)

In [ ]:
# Build Index
urls = [
    "https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-2-reflection/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-3-tool-use/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-4-planning/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-5-multi-agent-collaboration/?ref=dl-staging-website.ghost.io"
]

def load_and_split(urls, chunk_size=1000):
    doc_splits = []
    headers = {"User-Agent": "Mozilla/5.0"}

    for url in urls:
        try:
            response = requests.get(url, headers=headers, timeout=20)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            title = soup.title.string.strip() if soup.title and soup.title.string else "No Title"
            text = soup.get_text(separator=" ", strip=True)

            for i in range(0, len(text), chunk_size):
                chunk = text[i:i + chunk_size]
                if chunk.strip():
                    doc_splits.append({
                        "content": chunk,
                        "metadata": {
                            "title": title,
                            "source": url
                        }
                    })
        except Exception as e:
            print(f"Error loading {url}: {e}")

    return doc_splits

doc_splits = load_and_split(urls)
print(f"Loaded {len(doc_splits)} chunks")

In [ ]:
# Create embeddings
def get_embeddings(texts):
    result = genai.embed_content(
        model=embedding_model_name,
        content=texts,
        task_type="retrieval_document"
    )

    if isinstance(result, dict) and "embedding" in result:
        return np.array(result["embedding"])
    elif isinstance(result, dict) and "embeddings" in result:
        return np.array([item["values"] if isinstance(item, dict) and "values" in item else item for item in result["embeddings"]])
    else:
        raise ValueError(f"Unexpected embedding response format: {result}")

contents = [d["content"] for d in doc_splits]
doc_embeddings = get_embeddings(contents)
print(f"Embeddings shape: {doc_embeddings.shape}")

In [ ]:
# Retrieval function
def retrieve(query, docs, doc_embeddings, k=4):
    query_result = genai.embed_content(
        model=embedding_model_name,
        content=query,
        task_type="retrieval_query"
    )

    if isinstance(query_result, dict) and "embedding" in query_result:
        query_embedding = np.array(query_result["embedding"])
    elif isinstance(query_result, dict) and "embeddings" in query_result:
        first_item = query_result["embeddings"][0]
        query_embedding = np.array(first_item["values"] if isinstance(first_item, dict) and "values" in first_item else first_item)
    else:
        raise ValueError(f"Unexpected query embedding format: {query_result}")

    similarities = np.dot(doc_embeddings, query_embedding)
    top_indices = np.argsort(similarities)[-k:][::-1]
    return [docs[i] for i in top_indices]

### Question

In [ ]:
question = "What are the different kinds of agentic design patterns?"

### Retrieve docs

In [ ]:
retrieved_docs = retrieve(question, doc_splits, doc_embeddings)
print(f"Retrieved {len(retrieved_docs)} documents")

### Check what our doc looks like

In [ ]:
if retrieved_docs:
    doc = retrieved_docs[0]
    print(f"Title: {doc['metadata']['title']}\n")
    print(f"Source: {doc['metadata']['source']}\n")
    print(f"Content: {doc['content'][:500]}...\n")

### Check document relevancy

In [ ]:
def grade_documents(question: str, document: str) -> str:
    model = genai.GenerativeModel(llm_model_name)

    system_prompt = """You are a grader assessing relevance of a retrieved document to a user question.
If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant.
Return your response as a JSON object with a single key 'binary_score' whose value is either 'yes' or 'no'."""

    prompt = f"Retrieved document:\n\n{document}\n\nUser question:\n{question}"

    response = model.generate_content(
        [system_prompt, prompt],
        generation_config=genai.GenerationConfig(response_mime_type="application/json")
    )

    return json.loads(response.text).get("binary_score", "no")

### Filter out the non-relevant docs

In [ ]:
docs_to_use = []

for doc in retrieved_docs:
    print(f"Checking doc from: {doc['metadata']['source']}")
    score = grade_documents(question, doc['content'])
    print(f"Relevance Score: {score}")
    if score == 'yes':
        docs_to_use.append(doc)

print(f"Documents kept: {len(docs_to_use)}")

### Generate Result

In [ ]:
def generate_answer(question: str, docs: List[Dict]) -> str:
    model = genai.GenerativeModel(llm_model_name)

    context = "\n\n".join([
        f"Source: {d['metadata']['source']}\nContent: {d['content']}" for d in docs
    ])

    system_prompt = """You are an assistant for question-answering tasks.
Answer the question using the provided context.
Use three to five sentences maximum and keep the answer concise."""

    prompt = f"Retrieved documents:\n\n{context}\n\nUser question:\n{question}"
    response = model.generate_content([system_prompt, prompt])
    return response.text

generation = generate_answer(question, docs_to_use)
print("---" * 10)
print("GENERATED ANSWER:")
print(generation)

### Check for Hallucinations

In [ ]:
def grade_hallucinations(facts: str, generation: str) -> str:
    model = genai.GenerativeModel(llm_model_name)

    system_prompt = """You are a grader assessing whether an LLM generation is grounded in a set of retrieved facts.
Return a JSON object with key 'binary_score'.
'yes' means the answer is grounded in the facts, and 'no' means it is not."""

    prompt = f"Set of facts:\n\n{facts}\n\nLLM generation:\n{generation}"

    response = model.generate_content(
        [system_prompt, prompt],
        generation_config=genai.GenerationConfig(response_mime_type="application/json")
    )

    return json.loads(response.text).get("binary_score", "no")

facts_text = "\n\n".join([d['content'] for d in docs_to_use])
hallucination_score = grade_hallucinations(facts_text, generation)
print(f"Is grounded in facts: {hallucination_score}")

### Highlight used docs

In [ ]:
def highlight_documents(question: str, generation: str, docs: List[Dict]) -> Dict:
    model = genai.GenerativeModel(llm_model_name)

    docs_context = "\n\n".join([
        f"ID: doc{i+1}\nTitle: {d['metadata']['title']}\nSource: {d['metadata']['source']}\nContent: {d['content']}"
        for i, d in enumerate(docs)
    ])

    system_prompt = """You are an assistant for document grounding.
Identify the exact snippets from the provided documents that support the generated answer.
Return a JSON object with this structure:
{
  \"sources\": [
    {
      \"id\": \"doc ID\",
      \"title\": \"doc title\",
      \"source\": \"doc source url\",
      \"segment\": \"verbatim text snippet\"
    }
  ]
}
Only include documents actually used."""

    prompt = f"Used documents:\n\n{docs_context}\n\nUser question:\n{question}\n\nGenerated answer:\n{generation}"

    response = model.generate_content(
        [system_prompt, prompt],
        generation_config=genai.GenerationConfig(response_mime_type="application/json")
    )

    return json.loads(response.text)

lookup_response = highlight_documents(question, generation, docs_to_use)
print("---" * 10)
print("SOURCES HIGHLIGHTED:")
for item in lookup_response.get('sources', []):
    print(f"ID: {item['id']}\nTitle: {item['title']}\nSource: {item['source']}\nText Segment: {item['segment']}\n")